# BERT for Sentiment Analysis => fine-tuning a pre-trained BERT

In [1]:
#%pip install transformers -U

In [2]:
import transformers
import tensorflow as tf

# Downloading large review movie dataset (25000 reviews)

In [5]:
import urllib.request
file = './json_pol.json'

url = "https://thome.isir.upmc.fr/classes/RITAL/json_pol.json"
urllib.request.urlretrieve(url, file)
print(f"Téléchargé: {file}")

Téléchargé: ./json_pol.json


In [6]:
import json
from collections import Counter

# Loading json
file = './json_pol.json'
with open(file,encoding="utf-8") as f:
    data = json.load(f)


# Quick Check
counter = Counter((x[1] for x in data))
print("Number of reviews : ", len(data))
print("----> # of positive : ", counter[1])
print("----> # of negative : ", counter[0])
print("")
print(data[0])


Number of reviews :  25000
----> # of positive :  12500
----> # of negative :  12500

['Although credit should have been given to Dr. Seuess for stealing the story-line of "Horton Hatches The Egg", this was a fine film. It touched both the emotions and the intellect. Due especially to the incredible performance of seven year old Justin Henry and a script that was sympathetic to each character (and each one\'s predicament), the thought provoking elements linger long after the tear jerking ones are over. Overall, superior acting from a solid cast, excellent directing, and a very powerful script. The right touches of humor throughout help keep a "heavy" subject from becoming tedious or difficult to sit through. Lastly, this film stands the test of time and seems in no way dated, decades after it was released.', 1]


# Getting the Tokenizer

In [7]:
#from transformers import DistilBertForSequenceClassification, DistilBertTokenizer
#tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-cased')
model_name = "haisongzhang/roberta-tiny-cased"
#model_name = "bert-base-cased"

#model_name = "distilbert-base-uncased-finetuned-yelp-polarity"
#model_name = "textattack/bert-base-uncased-yelp-polarity"
#model_name = "rttl-ai/bert-base-uncased-yelp-reviews"


from transformers import AutoTokenizer#, BertForSequenceClassification
tokenizer = AutoTokenizer.from_pretrained(model_name)


config.json:   0%|          | 0.00/460 [00:00<?, ?B/s]

c:\Users\pc cam\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pc cam\.cache\huggingface\hub\models--haisongzhang--roberta-tiny-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

# Experiment the Tokenizer on the first train review

In [9]:
maxL = 512 # Max length of the sequence

string_tokenized = tokenizer(data[0][0], return_tensors="pt",
                                        add_special_tokens=True,  # add '[CLS]' and '[SEP]'
                            max_length=maxL,  # set max length
                            truncation=True,  # truncate longer messages
                            #pad_to_max_length=True
                            padding='max_length',  # add padding
                            return_attention_mask=True)

The output of the tokenizer string_tokenized (class BatchEncoding) returns two elements:


*   string_tokenized['input_ids']: the index of each token in the dictionary
*   string_tokenized['attention_mask']: a binary mask (0 to ignore the token, 1 to consider it). This is because we need tensor a fixed length and we have reviews with a variable number of words



In [10]:
print(string_tokenized['input_ids'])
print(string_tokenized['attention_mask'])

tensor([[  101,  1966,  4755,  1431,  1138,  1151,  1549,  1106,  1987,   119,
         22087, 10589,  1116,  1111, 11569,  1103,  1642,   118,  1413,  1104,
           107, 19923, 25945,  1279,  1109, 25861,   107,   117,  1142,  1108,
           170,  2503,  1273,   119,  1135,  4270,  1241,  1103,  6288,  1105,
          1103,  1107,  7854, 18465,   119,  4187,  2108,  1106,  1103, 10965,
          2099,  1104,  1978,  1214,  1385,  6193,  1985,  1105,   170,  5444,
          1115,  1108, 13493,  1106,  1296,  1959,   113,  1105,  1296,  1141,
           112,   188,  3073, 13328, 11462,   114,   117,  1103,  1354,  5250,
         20202,  3050,   181,  7728,  1263,  1170,  1103,  7591, 26525,  3200,
          1132,  1166,   119,  8007,   117,  7298,  3176,  1121,   170,  4600,
          2641,   117,  6548, 10404,   117,  1105,   170,  1304,  3110,  5444,
           119,  1109,  1268, 13193,  1104,  8594,  2032,  1494,  1712,   170,
           107,  2302,   107,  2548,  1121,  2479, 2

**You can use the BERT model for directly predicting polarity.** Let us apply that on the first review which has been tokenized with string_tokenized.

# Let's tokenize the whole dataset

In [12]:
import numpy as np

maxL = 512


inputs_tokens = []
attention_masks = []

for i in range(len(data)):
    if(i%2500==0):
        print(i)
    string_tokenized = tokenizer(data[i][0], return_tensors="pt",
                                        add_special_tokens=True,  # add '[CLS]' and '[SEP]'
                            max_length=maxL,  # set max length
                            truncation=True,  # truncate longer messages
                            #pad_to_max_length=True
                            padding='max_length',  # add padding
                            return_attention_mask=True)

    inputs_tokens.append(string_tokenized['input_ids'])
    attention_masks.append(string_tokenized['attention_mask'])

0
2500
5000
7500
10000
12500
15000
17500
20000
22500


# Let's create a 'TensorDataSet' FOR THE SAMPLES where each element is a triplet composed of token word index, token mask, and label

In [ ]:
# print(data[1300][1])
# print(y[0])

In [13]:
import torch
# Converting input tokens to torch tensors
inputs_tokens = torch.cat(inputs_tokens, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)


In [14]:
# Converting labels to torch tensor
#y = torch.zeros((len(data),2), dtype=torch.float)
y = torch.zeros((len(data),), dtype=torch.float)

for i in range(len(data)):
    y[i] = data[i][1]
    #y[i][data[i][1]] = 1
#y = torch.from_numpy(y)

In [15]:
print(inputs_tokens.shape)
print(attention_masks.shape)
print(y.shape)

torch.Size([25000, 512])
torch.Size([25000, 512])
torch.Size([25000])


In [16]:
from sklearn.model_selection import train_test_split

np.random.seed(0)
rs=10

inputs_tokens_train, inputs_tokens_test, attention_masks_train, attention_masks_test, y_train, y_test =train_test_split(inputs_tokens, attention_masks, y, test_size=0.5, random_state=rs)

print(inputs_tokens_train.shape)
print(inputs_tokens_test.shape)

print(attention_masks_train.shape)
print(attention_masks_test.shape)

print(y_train.shape)
print(y_test.shape)

torch.Size([12500, 512])
torch.Size([12500, 512])
torch.Size([12500, 512])
torch.Size([12500, 512])
torch.Size([12500])
torch.Size([12500])


In [ ]:
#print(y_train[10000])

In [17]:
from torch.utils.data import TensorDataset, random_split, DataLoader, RandomSampler, SequentialSampler

dataset_train = TensorDataset(inputs_tokens_train,  attention_masks_train, y_train)
dataset_test = TensorDataset(inputs_tokens_test,  attention_masks_test, y_test)

# Lets download a BERT model for word embedding

In [18]:
from transformers import BertForSequenceClassification
model = BertForSequenceClassification.from_pretrained(model_name)

pytorch_model.bin:   0%|          | 0.00/112M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: haisongzhang/roberta-tiny-cased
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/112M [00:00<?, ?B/s]

In [19]:
print(model)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 512, padding_idx=0)
      (position_embeddings): Embedding(512, 512)
      (token_type_embeddings): Embedding(2, 512)
      (LayerNorm): LayerNorm((512,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-3): 4 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=512, out_features=512, bias=True)
              (key): Linear(in_features=512, out_features=512, bias=True)
              (value): Linear(in_features=512, out_features=512, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=512, out_features=512, bias=True)
              (LayerNorm): LayerNorm((512,), eps=1e-12, e

In [ ]:
#train_dataloader = DataLoader(dataset_train, batch_size=2,shuffle=True)

#it, tm, labelsb = next(iter(train_dataloader))
#
#pred = model(input_ids=it, attention_mask=tm)
#print(pred[0])

In [ ]:
#print(pred.logits)

# FINE-TUNING THE MODEL

In [20]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [21]:
def accuracy(model, dataloader):
  model.eval()
  nbgood =0
  for idx,batch in enumerate(dataloader):
    b_input_ids = batch[0].cuda()
    b_input_mask = batch[1].cuda()
    b_labels = batch[2].cuda()

    with torch.no_grad():
      pred = model(input_ids=b_input_ids, attention_mask=b_input_mask)
      yhat = pred.logits.argmax(axis=1)
      ytrue = b_labels
      nbgood += (yhat==ytrue).sum()

    if(idx>0 and idx%250==0):
      print("idx=",idx," nbgood=",nbgood.item())

  acc = nbgood / 125.0
  return acc.item()


In [22]:
def accuracy_batch(model, b_input_ids, b_input_mask, b_labels):
  model.eval()

  with torch.no_grad():
    pred = model(input_ids=b_input_ids, attention_mask=b_input_mask)
    yhat = pred.logits.argmax(axis=1)
    ytrue = b_labels

    print(yhat.detach().cpu().numpy())
    print(ytrue.detach().cpu().numpy())

    acc = (yhat==ytrue).sum()*4.0

  return acc.item()


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

tb = 25
train_dataloader = DataLoader(dataset_train, batch_size=tb, shuffle=True)
test_dataloader  = DataLoader(dataset_test, batch_size=tb, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device utilisé :", device)

nbepochs = 5
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

model.train()
model.to(device)

for e in range(nbepochs):
    for idx, batch in enumerate(train_dataloader):
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device).long()

        optimizer.zero_grad()

        outputs = model(input_ids=b_input_ids, attention_mask=b_input_mask)
        loss = criterion(outputs.logits, b_labels)

        loss.backward()
        optimizer.step()

        if idx > 0 and idx % 250 == 0:
            print(
                "** batch:", idx,
                "acc train=", accuracy(model, train_dataloader, device),
                "acc test=", accuracy(model, test_dataloader, device)
            )

    print(
        "**** epoch", e + 1,
        "acc train=", accuracy(model, train_dataloader, device),
        "acc test=", accuracy(model, test_dataloader, device)
    )